# 01 · Data Ingestion

Downloads / reads all raw data sources and writes parquet files to `data/raw/`.

**Sources**
| Source | Access | Output |
|--------|--------|--------|
| CEC DGStats | Bulk CSV (manual download) | `dgstats_raw.parquet` |
| CAISO OASIS | `gridstatus` API | `caiso_hourly_raw.parquet` |
| NREL DeepSolar CA | CSV | `deepsolar_ca_tracts.parquet` |
| Census ZIP–Tract crosswalk | CSV | `zip_tract_xwalk.parquet` |

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

RAW = ROOT / 'data' / 'raw'
RAW.mkdir(parents=True, exist_ok=True)

import pandas as pd
from data_ingestion import load_dgstats, fetch_caiso_all, load_deepsolar, load_zip_tract_crosswalk

## 1.1 CEC DGStats

Manual step: download the bulk CSV from the [CEC DGStats portal](https://www.energy.ca.gov/data-reports/energy-almanac/california-electricity-data/electric-utility-company-customer-data) and place it at `data/raw/dgstats_raw.csv`.

In [ ]:
dgstats_csv = RAW / 'dgstats_raw.csv'
assert dgstats_csv.exists(), f'Download DGStats CSV to {dgstats_csv}'

dgstats = load_dgstats(dgstats_csv)
print(f'DGStats rows after filters: {len(dgstats):,}')
dgstats.head(3)

In [ ]:
dgstats.to_parquet(RAW / 'dgstats_raw.parquet', index=False)
print('Saved dgstats_raw.parquet')

## 1.2 CAISO OASIS (via gridstatus)

Fetches 2015–2023 in monthly batches (~336 API calls). Each month cached as a parquet shard under `data/raw/caiso_shards/`.

In [ ]:
caiso = fetch_caiso_all(start_year=2015, end_year=2023)
print(f'CAISO rows: {len(caiso):,} | expected ~{8760*9:,} hours')
caiso.head(3)

In [ ]:
caiso.to_parquet(RAW / 'caiso_hourly_raw.parquet', index=False)
print('Saved caiso_hourly_raw.parquet')

## 1.3 NREL DeepSolar CA

Place `deepsolar_ca_tracts.csv` in `data/raw/` (available from Kaggle or the project's GitHub).

In [ ]:
ds_csv = RAW / 'deepsolar_ca_tracts.csv'
assert ds_csv.exists(), f'Place DeepSolar CSV at {ds_csv}'

deepsolar = load_deepsolar(ds_csv)
print(f'DeepSolar CA tracts: {len(deepsolar):,}')
deepsolar.to_parquet(RAW / 'deepsolar_ca_tracts.parquet', index=False)

## 1.4 ZIP–Tract Crosswalk

Download from HUD USPS ZIP-to-Tract crosswalk or IPUMS NHGIS. Place at `data/raw/zip_tract_xwalk.csv`.

In [ ]:
xwalk_csv = RAW / 'zip_tract_xwalk.csv'
assert xwalk_csv.exists(), f'Place crosswalk CSV at {xwalk_csv}'

xwalk = load_zip_tract_crosswalk(xwalk_csv)
print(f'ZIP-tract pairs (CA): {len(xwalk):,}')
xwalk.to_parquet(RAW / 'zip_tract_xwalk.parquet', index=False)

## Summary

| File | Rows |
|------|------|
| `dgstats_raw.parquet` | — |
| `caiso_hourly_raw.parquet` | — |
| `deepsolar_ca_tracts.parquet` | — |
| `zip_tract_xwalk.parquet` | — |

→ Proceed to `02_cleaning_qa.ipynb`